In [ ]:
!pip install -q pandas numpy torch transformers sentence-transformers faiss-cpu

In [ ]:
import pandas as pd
import numpy as np
import torch

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import faiss


In [ ]:
# Load dataset
df = pd.read_csv("/content/drive/MyDrive/NLP Project College/Information Retrieval NLP/Dataset/medquad.csv")

print("Dataset Shape:", df.shape)
df.head(10)


In [ ]:
# Remove missing values
df = df.dropna()

# Combine question + answer for better context
df["context"] = df["question"] + " " + df["answer"]

contexts = df["context"].tolist()


In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
embeddings = embedding_model.encode(
    contexts,
    batch_size=32,
    show_progress_bar=True
)

print("Embedding Shape:", embeddings.shape)


In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("Total vectors in index:", index.ntotal)


In [ ]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:
def medical_prompt(context, question):
    prompt = f"""
You are a medical information assistant.
Use the following medical context to answer the question accurately.
Do NOT add assumptions.

Medical Context:
{context}

Question:
{question}

Answer:
"""
    return prompt


In [ ]:
def retrieve_context(query, top_k=3):
    query_embedding = embedding_model.encode([query])
    distances, indices = index.search(query_embedding, top_k)

    retrieved_contexts = [contexts[i] for i in indices[0]]
    return "\n".join(retrieved_contexts)


In [ ]:
def generate_answer(question):
    context = retrieve_context(question)

    prompt = medical_prompt(context, question)

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)

    outputs = llm_model.generate(
        **inputs,
        max_length=200,
        temperature=0.3,
        do_sample=True
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer


In [ ]:
query = "What are the symptoms of diabetes?"

response = generate_answer(query)

print("User Query:", query)
print("\nMedical Answer:\n", response)


In [ ]:
from transformers import pipeline

# Load instruction-tuned LLM
qa_pipeline = pipeline(
    task="text-generation",
    model="NousResearch/Meta-Llama-3-8B-Instruct",
    device_map="auto"
)

def generate_answer(question):
    prompt = (
        "You are a helpful medical assistant.\n"
        "Give clear, simple, and general medical information.\n"
        "Do NOT diagnose diseases or prescribe medicines.\n"
        "If the issue seems serious, advise consulting a doctor.\n\n"
        f"Question: {question}\n"
        "Answer:"
    )

    response = qa_pipeline(
        prompt,
        max_new_tokens=180,
        temperature=0.5,
        do_sample=True
    )

    return response[0]["generated_text"].split("Answer:")[-1].strip()


# ---------------- MAIN PROGRAM ----------------
print("\n🩺 AI Medical Q&A System")
print("Type 'exit' to quit\n")
print("-" * 50)

try:
    while True:
        question = input("\nEnter your medical question: ").strip()

        if question.lower() == "exit":
            print("\n👋 Exiting Medical Q&A System. Stay healthy!")
            break

        print("\n⏳ Generating answer...\n")
        answer = generate_answer(question)

        print("💡 Medical Answer:")
        print(answer)
        print("-" * 50)

except KeyboardInterrupt:
    print("\n\n👋 Interrupted. Goodbye and take care!")
